# 25.43 Laboratorio de Procesamiento de Señales — TP2 Compresión de audio con wavelets

## El 95 % que se tira

En la primera clase el problema era una radiografía pesada  y una conexión de Internet rural: se puede tirar el 95 % de los datos sin que el diagnóstico cambie, y la pregunta es cuál 95 %. Este TP es esa pregunta, hecha sobre audio — donde el que decide si el diagnóstico cambió es **tu oído**.

La maquinaria es la del módulo: una base que localiza en tiempo y en escala, coeficientes que
caen en picada, y un solo gesto —quedarse con los grandes— que comprime y quita ruido según
por qué lo hagas.

Lo que este TP agrega sobre lo hecho en clase: **tus** audios, la comparación sistemática
entre familias y contenidos, y la defensa de cada elección con la curva y con los oídos.

---

## Qué se entrega

**Esta misma notebook, completada y corrida de arriba abajo**, más la carpeta `audios/` con tus
tres archivos y los comprimidos que el análisis cite. No hay informe aparte: el análisis va en
las celdas de markdown, al lado de la figura que lo sostiene.

Antes de entregar, *Kernel → Restart & Run All*. Si no corre de punta a punta, no se corrige.

**Si usás inteligencia artificial** —ChatGPT, Claude, Copilot o cualquier otra—, anexá el
registro de **todas** las preguntas que le hiciste: completas, en orden y sin editar. Va en un
archivo aparte junto a la notebook (`registro_IA.md`).

## Cómo se evalúa

| Criterio | Peso |
|---|---|
| **Análisis crítico** | **40 %** |
| Corrección técnica | 30 % |
| Figuras | 15 % |
| Reproducibilidad | 15 % |

**El núcleo son las cinco partes.** El núcleo bien hecho es un 8; las extensiones suman por
encima. Una curva bien discutida y una escucha bien documentada valen más que diez barridos
sin analizar.

> **Una advertencia sobre el método de trabajo.** Acá también vas a elegir parámetros —
> familia, nivel, fracción. La manera equivocada es probar hasta que suene bien y justificar
> después. La manera correcta es decidir primero **qué necesita cada contenido** —¿ataques
> secos? ¿armónicos que duran?— y recién entonces elegir, con la predicción anotada antes de
> escuchar. Se nota cuál de las dos hiciste.

## Las figuras

Ejes rotulados con unidades, escala elegida y un pie que diga qué hay que mirar. La curva
tasa–distorsión va en ejes logarítmicos en la fracción. Una figura sin pie no argumenta:
decora.

---

## Desde cuándo se puede hacer cada parte

| Parte | A partir de |
|---|---|
| 1. Tres audios, al microscopio | **C8 · 22/09** |
| 2. La curva tasa–distorsión | **C9 · 29/09** |
| 3. Elegí, y defendela | **C9 · 29/09** |
| 4. El test de escucha | **C10 · 06/10** |
| 5. El mismo acto: denoising | **C9 · 29/09** |
| Extensiones | cada una dice la suya |

Es **a partir de**, no *recién en*: si llegás antes por tu cuenta, adelante.

> **El 06/10, en C10, se hace el test de escucha de la parte 4**, contra referencias anónimas
> de cátedra: es lo único del TP que necesita a todos en el aula. Llegá con tus muestras de la
> parte 2 guardadas y tu predicción por escrito. Lo demás se trabaja en casa, y desde ahí
> queda una semana para cerrar bien.

## Los audios

Tres archivos tuyos, y los tres tipos son obligatorios porque la comparación entre familias
solo es informativa si el contenido las hace trabajar distinto:

| Tipo | Qué buscar | Ejemplo |
|---|---|---|
| **Música tonal** | armónicos sostenidos | una guitarra, un piano, un coro |
| **Voz** | lo tonal y lo percusivo mezclados | vos, leyendo un párrafo |
| **Transitorio percusivo** | ataques secos, silencio entre golpes | palmas, llaves, percusión |

Requisitos: WAV mono (o convertible), entre 5 y 15 segundos, sin clipping de origen. Material
libre sirve (Freesound, Wikimedia Commons); grabado por vos, mejor. Nombralos en la celda de
carga y guardalos en `audios/` junto a esta notebook.

In [3]:
# inline y no widget: la entrega tiene que renderizar sola para quien la corrija
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import pywt
from scipy.io import wavfile
from IPython.display import Audio, display

plt.rcParams["figure.figsize"] = (11, 3.6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print("Listo. pywt", pywt.__version__)

Listo. pywt 1.8.0


---

## Carga de los audios

La función está resuelta: leer un WAV no es lo que este TP evalúa. **Qué audio elegiste, sí** —
un tonal que es una senoidal pura, o un «percusivo» que es una canción entera, desarman la
comparación que el TP necesita.

Completá el diccionario con tus tres archivos.

In [5]:
def cargar(ruta):
    """WAV → (x, fs), con x en float, mono, normalizado a |x| ≤ 1."""
    fs, x = wavfile.read(ruta)
    x = x.astype(float)
    if x.ndim == 2:
        x = x.mean(axis=1)
    x /= np.abs(x).max()
    return x, fs


AUDIOS = {
    # completá con tus tres archivos:
    "tonal": "audios/tonal.wav",
    "voz": "audios/voz.wav",
    "percusivo": "audios/percusivo.wav",
}

señales = {}
for nombre, ruta in AUDIOS.items():
    x, fs = cargar(ruta)
    señales[nombre] = (x, fs)
    print(f"{nombre:10s} {ruta}  —  {len(x)/fs:.1f} s a {fs} Hz")
    display(Audio(x, rate=fs))

tonal      audios/tonal.wav  —  14.3 s a 48000 Hz


voz        audios/voz.wav  —  14.2 s a 48000 Hz


percusivo  audios/percusivo.wav  —  11.0 s a 48000 Hz


---

# Parte 1 — Tres audios, al microscopio

> **A partir de C8 · 22/09**

Antes de comprimir nada: **medí la sparsity** de cada audio en cada base, que es lo que decide
todo lo demás.

**Qué hay que hacer.**

1. Para cada audio y para al menos tres familias (Haar, una Daubechies, CDF 9/7 = `bior4.4`):
   la DWT multinivel, los coeficientes ordenados de mayor a menor en `semilogy`, y el error de
   quedarse con $M$ — los dos paneles del laboratorio de C8, ahora sobre tus datos.
2. **Elegí el nivel de descomposición $J$ y defendelo.** No es un default: con $J$ chico la
   aproximación se lleva casi toda la energía y el barrido de la parte 2 no decide nada; con
   $J$ grande estás analizando escalas más largas que tus notas. Mostrá qué cambia entre dos
   valores de $J$ y decí con cuál seguís.
3. Agregá la DCT como referencia: es la base que no localiza.

**Qué hay que responder.**

- ¿Qué audio es más sparse, y en qué base? ¿Coincide con lo que el contenido hace esperar?
- ¿Dónde viven los coeficientes grandes del percusivo? Mostralo con la figura de niveles del
  laboratorio de C8.
- ¿Para qué audio la DCT compite mejor contra las wavelets, y por qué justamente ése?

In [ ]:
# Parte 1 — tu código acá


*Tu análisis de la parte 1, acá.*

---

# Parte 2 — La curva tasa–distorsión

> **A partir de C9 · 29/09**

La figura central del TP: **familias × fracciones × contenidos**, medida.

**Qué hay que hacer.**

1. Comprimí cada audio reteniendo los $M$ coeficientes más grandes, para fracciones que cubran
   de 50 % a 1 % — las cuatro del laboratorio son el esqueleto; agregá puntos donde tu curva
   cambie rápido.
2. Medí la SNR de cada reconstrucción, y armá la curva tasa–distorsión por familia, un panel
   por audio, todas las familias en los mismos ejes.
3. Guardá los comprimidos que vas a escuchar en la parte 4 (`wavfile.write`, normalizando
   antes para no clipear).

**Qué hay que responder.**

- ¿Qué familia gana en cada contenido, y el orden cambia entre contenidos?
- ¿Dónde está la rodilla de cada curva — la fracción debajo de la cual el costo se dispara?
- La SNR del percusivo, ¿cuenta lo que le pasa a los ataques? Medí la SNR en ventanas cortas
  alrededor de un ataque y comparala con la global. Guardá esa observación: la parte 4 la
  retoma.

In [ ]:
# Parte 2 — tu código acá


*Tu análisis de la parte 2, acá. Incluí la justificación del rango de fracciones que barriste.*

---

# Parte 3 — Elegí, y defendela

> **A partir de C9 · 29/09**

Esta parte casi no tiene código, y es la que más pesa.

Para cada uno de tus tres audios, elegí **una familia y una fracción** para dos usos
distintos: (a) *transparencia* — que no se note—, y (b) *la máxima compresión que tolerarías*
para mandar ese audio por un canal malo. Son seis decisiones, y cada una se defiende con la
curva de la parte 2 **y** con tu escucha.

**Qué hay que responder.**

- Para cada decisión: qué medición la sostiene, y qué escuchaste que el número no decía.
- ¿En qué contenido la SNR y tu oído estuvieron más en desacuerdo? ¿Por qué ahí?
- ¿Hay alguna elección donde familias distintas empatan en SNR pero no en escucha? Ésa es la
  más valiosa: documentala con los dos audios guardados.

> Las elecciones pueden diferir de las de otro grupo y estar las dos bien. Lo que se corrige
> es el argumento: una elección defendida con una medición y una escucha documentada vale; la
> misma elección defendida con «sonaba bien», no.

In [ ]:
# Parte 3 — tu código acá (poco: los seis puntos elegidos, marcados sobre las curvas)


*Tu defensa, acá. Es la sección más importante de la entrega.*

---

# Parte 4 — El test de escucha

> **A partir de C10 · 06/10 — se hace en clase**

En C10 el docente arma el set de escucha: las muestras de los dos grupos **más
referencias anónimas de cátedra** — el mismo audio a varias fracciones, con implementación
propia — de modo que nadie pueda mapear muestra → autor. Cada uno anota, por muestra: ¿se
degradó respecto del original? ¿En qué se nota?

**Qué hay que hacer acá.**

1. Antes de C10: dejá listas tus muestras (parte 2) y **tu predicción por escrito** de
   cuáles van a sobrevivir la escucha.
2. Después de la escucha: volcá los resultados — aciertos, sorpresas, y el contraste entre lo que
   la SNR decía y lo que el conjunto escuchó.

**Qué hay que responder.**

- ¿Qué fracción resultó «transparente» según la escucha, y coincide con tu predicción de la
  parte 3?
- ¿Qué artefacto se identificó más fácil — el emborronado de un ataque, el burbujeo en lo
  tonal— y en qué familia?
- Si la escucha y la SNR ordenaron distinto a dos familias: ¿cuál de las dos medidas le
  creerías para *este* uso, y por qué?

In [ ]:
# Parte 4 — tu código acá (la tabla de resultados de la escucha, y su figura si aporta)


*Tu análisis de la parte 4, acá.*

---

# Parte 5 — El mismo acto: denoising

> **A partir de C9 · 29/09**

La clase afirmó que comprimir y quitar ruido son el mismo acto. Acá lo mostrás sobre tu audio.

**Qué hay que hacer.**

1. Tomá tu audio tonal, sumale ruido blanco con un $\sigma$ que elijas y declares (que se
   oiga, sin tapar la música).
2. Denoiseá con la cadena de la clase: umbral universal, $\hat{\sigma}$ por MAD sobre el
   nivel más fino, umbral blando.
3. Compará $\hat{\sigma}$ con tu $\sigma$, mostrá antes/después —forma de onda y
   escucha— y medí la SNR ganada.

**Qué hay que responder.**

- ¿Qué coeficientes murieron acá, y qué coeficientes morían al comprimir? ¿En qué se
  diferencia *por qué* los tirás en cada caso?
- ¿Qué le hizo el umbral a los ataques, si tu audio los tenía? ¿Se compra gratis el silencio?

In [ ]:
# Parte 5 — tu código acá


*Tu análisis de la parte 5, acá.*

---

# Extensiones

Suman por encima del núcleo. **Una hecha bien vale más que tres empezadas.** No hace falta
ninguna para aprobar cómodo.

**A. La transformada que no decima** · *a partir de C9*
La DWT decima, y por eso el resultado depende de dónde empieza la señal. La versión
estacionaria (SWT, `pywt.swt`) no decima: denoiseá con las dos y comparalas **cerca de los
ataques**. ¿Qué artefacto desaparece, y qué precio se paga en cuentas y en memoria?

**B. La mejor base, por entropía** · *a partir de C8*
Wavelet packets sobre tu percusivo: dejá que `pywt.WaveletPacket` abra el árbol completo y
elegí la mejor base por entropía. ¿Le gana a la DWT en la curva tasa–distorsión? ¿Y en la
escucha?

**C. Bits de verdad** · *a partir de C9*
«Retener el 5 %» no es una tasa en bits: cuantizá los coeficientes retenidos (elegí y declará
los bits por coeficiente) y estimá la tasa real en kbit/s contando también dónde están los
retenidos. Rehacé la curva tasa–distorsión con kbit/s en el eje x. ¿Cambia el orden de las
familias?

**D. La radiografía, de verdad** · *a partir de C9*
El problema dos de C1 era una imagen. Tomá una imagen en escala de grises (una radiografía de
dominio público, por ejemplo), comprimila con `bior4.4` en 2D (`pywt.wavedec2`) al 5 % y al
1 %, y mostrá dónde vive el error. Es JPEG-2000 en miniatura — decilo con una figura.

---

## Antes de entregar

- *Kernel → Restart & Run All*, y que corra entero — con tus WAV en `audios/`, que viajan con
  la entrega.
- Todas las figuras con ejes rotulados, unidades y pie.
- Las celdas de análisis contestadas, no vacías. **Son el 40 %.**
- Nombres de los integrantes en la celda de abajo.

**Los seis errores que más aparecen.**

1. **Elegir audios que no comprometen.** Una senoidal pura como «tonal» hace ganar a la DCT y
   no dice nada de wavelets; una canción entera como «percusivo» mezcla todo y no separa nada.
2. Comparar familias con $J$ distinto o con señales de largo distinto, y leer la diferencia
   como si fuera de la familia.
3. Medir solo la SNR global en el percusivo: el daño vive en los ataques y el promedio lo
   esconde.
4. Guardar los WAV comprimidos sin renormalizar y escuchar **clipping** como si fuera un
   artefacto de la wavelet.
5. Escuchar con los parlantes del notebook y concluir «no hay diferencia». Auriculares.
6. Elegir la fracción por lo que suena bien y escribir la justificación después — la
   predicción va **por escrito, antes** de escuchar.

---

**Grupo:**

**Integrantes:**

**Extensiones hechas:**